# Query routing

<img src="query_routing.png">

## Prepare the data

我们使用 Langchain WebBaseLoader 从博客源加载文档，并通过 RecursiveCharacterTextSplitter 将其拆分为多个片段。

In [14]:
import os

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [15]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a WebBaseLoader instance to load documents from web sources
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        ),# 只解析 HTML 中符合特定条件的部分
    )
)

# Load documents from web sources using the loader
documents=loader.load()

# Initialize a RecursiveCharacterTextSplitter for splitting text into chunks
text_spliiter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)

# Split the documents into chunks using the text_splitter
docs=text_spliiter.split_documents(documents)

# Inspect
docs[1]

Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Short-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.\nLong-term memory: This provides the agent with the capability to retain and recall (infinite) information over extended periods, often by leveraging an external vector store and fast retrieval.\n\n\nTool use\n\nThe agent learns to call external APIs for extra information that is missing from the model weights (often hard to change after pre-training), including current information, code execution capability, access to proprietary information sources and more.\n\n\n\n\n\nOverview of a LLM-powered autonomous agent system.')

## Build the chain

We load the docs into milvus vectorstore, and build a milvus retriever.

In [16]:
from rag_utils.vanilla import vectorstore,format_docs,rag_prompt,llm

vectorstore.add_documents(docs)
retriever = vectorstore.as_retriever()

Define the vanilla RAG chain.

In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import PromptTemplate

vanilla_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

Define the sub query chain.

In [18]:
from rag_utils.sub_query import SubQueryRetriever

sub_query_retriever=SubQueryRetriever.from_vectorstore(vectorstore)

sub_query_chain=(
    {"context": sub_query_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

构建一个路由器链，并尝试调用它。它可以返回一个字符串，用于判断查询是否可分解。

In [19]:
from rag_utils.route import ROUTER_PROMPT

router_chain=(
    {"question":RunnablePassthrough()}
    |PromptTemplate.from_template(ROUTER_PROMPT)
    |llm
    |StrOutputParser()
)

In [20]:
router_chain.invoke("How can I use Milvus and what is the zilliz")

'Classification: Decomposable\nReason: The question can be decomposed into two sub-questions: "How can I use Milvus?" and "What is Zilliz?".'

Define a route function.

In [21]:
from rag_utils.route import parse_router_output

def route(info):
    if parse_router_output(info["category"])=="Decomposable":
        print("invoke sub_query_chain")
        return RunnableLambda(lambda x:x["question"])|sub_query_chain
    else:
        print("invoke vanilla_rag_chain")
        return RunnableLambda(lambda x:x["question"])|vanilla_rag_chain

Let's define the full chain.

In [22]:
full_chain={
    "category":router_chain,
    "question":RunnablePassthrough(),
}|RunnableLambda(route)

## Test the chain

In [23]:
# 有哪些不同类型的内存以及不同的神经网络算法？
query1 = "Which are the different types of memory and different types ANN algorithms?"

print("\n\n",full_chain.invoke(query1))

invoke sub_query_chain
sub_queries: ['Sub-questions:', 'What are the different types of memory?', 'What are the different types of ANN algorithms?']


 The different types of memory mentioned are:

- **Sensory Memory**: lasts up to a few seconds; includes iconic (visual), echoic (auditory), and haptic (touch) memory.
- **Short-Term Memory (STM) / Working Memory**: holds about 7 items and lasts 20–30 seconds.
- **Long-Term Memory (LTM)**: can last from days to decades with essentially unlimited capacity. It includes:
  - **Explicit / declarative memory**: episodic memory (events/experiences) and semantic memory (facts/concepts).
  - **Implicit / procedural memory**: unconscious skills and routines, like riding a bike.

The different ANN algorithms for fast Maximum Inner Product Search (MIPS) mentioned are:

- **FAISS** (Facebook AI Similarity Search): uses vector quantization by partitioning the vector space into clusters, then refines quantization within clusters.
- **ScaNN** (Scalable

In [24]:
print(vanilla_rag_chain.invoke(query1))

The context identifies these types of memory:

- **Sensory memory** — includes **iconic memory** (visual), **echoic memory** (auditory), and **haptic memory** (touch)
- **Short-term memory** — described as in-context learning, limited by the Transformer’s finite context window
- **Long-term memory** — described as an external vector store accessible via fast retrieval

The context mentions that common choices of **ANN algorithms** for fast MIPS exist, but it does **not list any specific ANN algorithm names**. Therefore, I cannot provide the different ANN algorithm types from the provided context.


## 回答质量对比
|方法	|回答内容	|评价|
|---|---|---|
|Vanilla RAG	|仅识别了记忆类型（感官、短期、长期），但对于 ANN 算法部分，只说了“常见选择存在，但未列出具体名称”，无法提供任何具体算法。	|❌ 检索到的文档可能只包含了记忆相关的信息，而缺少关于 ANN 算法的具体列举，导致回答不完整。
|Sub Query RAG	|分别拆解了“不同类型记忆”和“不同类型 ANN 算法”两个子问题，检索到了记忆的详细分类（感官、短时、长时及其子类），以及 ANN 算法的具体名称（FAISS、ScaNN），并给出了完整、条理清晰的回答。	|✅ 通过子问题分解，分别检索，覆盖了问题的两方面，确保了两部分信息都得到准确回答，内容全面、具体。|

In [25]:
# query2 = “有哪些不同类型的记忆？”
query2 = "Which are the different types of memory?"

print("\n\n", full_chain.invoke(query2))

invoke vamolla_rag_chain


 The different types of memory described are:

- **Sensory Memory**: The earliest stage; retains sensory impressions after stimuli end. It typically lasts only up to a few seconds. Subcategories include iconic (visual), echoic (auditory), and haptic (touch) memory.

- **Short-Term Memory (STM) / Working Memory**: Holds information currently being used for complex cognitive tasks. It has a capacity of about **7 items** (Miller 1956) and lasts for **20–30 seconds**.

- **Long-Term Memory (LTM)**: Stores information from a few days to decades, with essentially unlimited capacity. It has two subtypes:
  - **Explicit / Declarative Memory**: Conscious recall of facts and events, including episodic memory (events/experiences) and semantic memory (facts/concepts).
  - **Implicit / Procedural Memory**: Unconscious memory for skills and routines, such as riding a bike or typing.


In [26]:
print(vanilla_rag_chain.invoke(query2))

The different types of memory described are:

- **Sensory Memory**: The earliest stage; retains sensory impressions for up to a few seconds. Subtypes include iconic (visual), echoic (auditory), and haptic (touch).
- **Short-Term Memory (STM) / Working Memory**: Holds information currently in awareness for complex cognitive tasks. It has a capacity of about **7 items** (Miller 1956) and lasts **20–30 seconds**.
- **Long-Term Memory (LTM)**: Stores information from a few days to decades, with essentially unlimited capacity. It has two subtypes:
  - **Explicit / declarative memory**: Consciously recalled facts and events, including episodic memory and semantic memory.
  - **Implicit / procedural memory**: Unconscious skills and routines, like riding a bike or typing.


## 回答质量对比
|方法	|回答内容	|评价|
|---|---|---|
|Vanilla RAG	|详细列出了感官记忆（含子类）、短时记忆（容量和时长）、长时记忆（分显性和隐性）等，结构清晰，信息完整。	|✅ 对于单一主题的提问，检索结果充分，回答准确、详尽，没有遗漏。
|Sub Query RAG	|同样给出了上述完整的记忆分类和描述，内容与 Vanilla 基本一致	|✅ 对于单一主题（仅问记忆），Sub Query 也能检索到相同的完整信息，回答质量与 Vanilla 相当|